[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/02_preprocessing.ipynb)

# 02 — Preprocessing

**Purpose.** Turn the raw simulation output into the two verified, typed tables the
rest of the pipeline reads. Nothing here learns anything from the data.

**Inputs.** The scenario, radio-map and MDT artifacts at the paths in
`cfg.simulation.output`, read-only. The demand map is not an input here.

**Outputs.**
- `data/processed/mdt.parquet` — the synthetic MDT, verified and typed: a reported
  RSRP (`rsrp_*`) and SINR (`sinr_*`) per cell-band, and one explicit reported
  indicator (`reported_*`) per cell-band covering both, since SINR has a path
  exactly where RSRP does.
- `data/processed/cell.parquet` — the cell-band table the radio map was solved at:
  the pre-optimization configuration every `DeltaTilt` is measured against, with
  each cell-band's `max_prb` limit.

---

### The boundary rule

**Allowed here** (deterministic, model-agnostic):
- Schema and dtype verification
- Hard bounds whose source can be *named* — the config, or the scenario manifest
- Typing, column ordering, and a stable row order

**Forbidden here** (learns from the data — belongs in a `02x` notebook, fitted on
train only):
- Imputation of the censored measurements
- Statistical outlier detection
- Scaling, encoding, any fitted transformation
- Feature engineering

Two rules this notebook inherits from `01_eda.ipynb` section 11, both of which it
would be easy to get wrong:

- An empty `rsrp_*` or `sinr_*` means *no path* **or** *not reported*, and the file cannot tell
  the two apart. It is never read as a zero, and never imputed here.
- The −120 / −90 dBm KPI thresholds **classify a tile**. They do not filter a row.
  Holes are the measurement, not a defect.

**This notebook is a thin wrapper over `src/data/`.** The same sequence runs
unattended as `uv run python -m src.data.build`. Logic in the module, narration
here.

## 0. Environment

Run this section first, wherever you are.

**Locally** it only walks up to the project root and makes it the working directory,
so the root-relative paths in `configs/` resolve the same way they do for
`task simulation`. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path` so `import src`
works without an editable install, and installs the one package Colab does not
already ship. No Sionna-RT here: this notebook reads artifacts, it does not
ray-trace. Note that `data/` is DVC-tracked and therefore *not* part of the clone —
see the Drive cell below.

In [1]:
# --- Environment bootstrap -------------------------------------------------
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = "."  # project root inside the repository

# (import name, pip name). Colab already ships numpy, pandas, pyarrow, matplotlib
# and seaborn, so only this one is installed. Sionna-RT is deliberately absent:
# nothing here loads a scene.
COLAB_PACKAGES = [("hydra", "hydra-core")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = (checkout / SUBDIR).resolve()
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1.
CONFIG_OVERRIDES: list[str] = []

print(f"project root: {root}   colab: {IN_COLAB}")

project root: d:\Projects\band-tilt   colab: False


In [ ]:
# --- Colab: the artifacts (optional) ---------------------------------------
# data/ is DVC-tracked, so it is not in the Git clone and a fresh Colab runtime has
# none of it. Either run 00_simulation.ipynb first, or mount Drive and point the
# config at a copy there — Drive also survives a runtime reset, which /content does
# not. This notebook *writes* the two processed tables, so without Drive they are
# lost when the runtime ends.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# DATA_ROOT = "/content/drive/MyDrive/band-tilt/data"
# CONFIG_OVERRIDES += [
#     f"simulation.output.ue_file={DATA_ROOT}/external/ue_positions.csv",
#     f"simulation.output.manifest_file={DATA_ROOT}/external/scenario.json",
#     f"simulation.output.radio_map_file={DATA_ROOT}/interim/radio_map.npz",
#     f"simulation.output.mdt_file={DATA_ROOT}/interim/mdt.csv",
#     f"simulation.output.demand_map_file={DATA_ROOT}/interim/demand_map.npz",
#     f"data.output.mdt_file={DATA_ROOT}/processed/mdt.parquet",
#     f"data.output.cell_file={DATA_ROOT}/processed/cell.parquet",
# ]

## 1. Setup

Compose the config, seed everything, and import from `src/`. Every notebook starts
the same way so that a cell copied between notebooks behaves identically.

In [3]:
%load_ext autoreload
%autoreload 2

from functools import partial

import numpy as np
import pandas as pd

from src.config import load_config
from src.evaluation.export import save_table
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()

# Tables land here so they can be read without rerunning the notebook; skipped on Colab.
save_table = partial(save_table, in_colab=IN_COLAB, directory=Path("reports/tables/02_preprocessing"))

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

print(f"mdt  -> {cfg.data.output.mdt_file}")
print(f"cell -> {cfg.data.output.cell_file}")

mdt  -> data/processed/mdt.parquet
cell -> data/processed/cell.parquet


## 2. Load the artifacts

Read-only, and all four together: the MDT alone cannot be verified, because what it
is checked *against* lives in the radio map and the manifest. If a file is missing,
run `00_simulation.ipynb` or `task simulation`.

In [4]:
from src.data.load import load_artifacts

artifacts = load_artifacts(cfg)
n_rows, n_cols = artifacts.shape

pd.DataFrame(
    {
        "value": {
            "scenario_id": artifacts.scenario_id,
            "mdt rows": len(artifacts.mdt),
            "mdt columns": artifacts.mdt.shape[1],
            "ue rows": len(artifacts.ue),
            "cells": len(artifacts.tx_names),
            "bands": len(artifacts.band_labels),
            "cell-band pairs": len(artifacts.measurement_columns),
            "grid": f"{n_rows} x {n_cols} tiles",
        }
    }
)

,value
scenario_id,scn_b2779850236462be
mdt rows,9701
mdt columns,43
ue rows,9939
cells,12
bands,3
cell-band pairs,36
grid,61 x 74 tiles


The MDT is shorter than the UE table by design — `mdt.build` drops UEs no
transmitter reaches — so the two cannot be joined by row order. Section 11 of
`01_eda.ipynb` names `(t_index, x, y)` as the join key, and section 3 below checks
that it is unique.

## 3. Verify against the contract

Fail here rather than three notebooks later. Every check names the **source** of the
bound it enforces: a value in `configs/simulation.yaml`, a value in the scenario
manifest, or the radio map itself. A bound with no nameable source would be a
statistical threshold, and belongs in `02x` fitted on train only.

Note what is *not* checked. The KPI thresholds classify a tile as a hole or as weak;
they never disqualify a measurement, so they cannot appear here.

In [5]:
from src.data import schema

checks = schema.verify(artifacts, cfg)
save_table(checks, "verification_checks")
checks


,check,source,holds,violations
0,npz tx_name matches the configured cells,configs/simulation.yaml,True,0
1,npz band_label matches the configured bands,configs/simulation.yaml,True,0
2,npz scenario_id matches the manifest,scenario.json,True,0
3,npz grid matches the manifest grid,scenario.json,True,0
4,npz ue_height_m matches the config,configs/simulation.yaml,True,0
5,mdt position columns are the declared set,configs/simulation.yaml,True,0
6,"mdt measurement columns are cell x band, in order",radio_map.npz,True,0
7,z equals the configured UE height,configs/simulation.yaml,True,0
8,0 <= tile_row < n_rows,scenario.json,True,0
9,0 <= tile_col < n_cols,scenario.json,True,0


In [6]:
schema.require(checks)  # raises SchemaError listing anything that failed
print(f"all {len(checks)} checks passed")

all 19 checks passed


## 4. The cell configuration

One row per cell-band pair — the decision variable, at its
baseline. This is the configuration the radio map was solved at, so it is the
"before" that every `DeltaTilt` an optimizer proposes will be measured against.

`rsrp_column` and `sinr_column` are the join keys back to `mdt.parquet`: each row
names the measurement columns that carry this cell-band's reports. `max_prb` is the
PRB limit the serving rule holds this cell-band to.

In [7]:
from src.data.build import build_cells

cells = build_cells(cfg, artifacts)
print(f"{len(cells)} cell-band pairs over {cells['node'].nunique()} nodes")
cells

36 cell-band pairs over 4 nodes


,cell,node,x,y,z,azimuth_deg,band,frequency_hz,bandwidth_hz,tilt_baseline_deg,tilt_min_deg,tilt_max_deg,rsrp_column,scenario_id
0,n0c0,n0,-380.34,-489.80,30.0,0.0,b2600,2.600000e+09,100000000.0,8.0,0.0,16.0,rsrp_n0c0_b2600,scn_b2779850236462be
1,n0c0,n0,-380.34,-489.80,30.0,0.0,b1800,1.800000e+09,100000000.0,6.0,0.0,14.0,rsrp_n0c0_b1800,scn_b2779850236462be
2,n0c0,n0,-380.34,-489.80,30.0,0.0,b700,7.000000e+08,100000000.0,4.0,0.0,12.0,rsrp_n0c0_b700,scn_b2779850236462be
3,n0c1,n0,-380.34,-489.80,30.0,120.0,b2600,2.600000e+09,100000000.0,8.0,0.0,16.0,rsrp_n0c1_b2600,scn_b2779850236462be
4,n0c1,n0,-380.34,-489.80,30.0,120.0,b1800,1.800000e+09,100000000.0,6.0,0.0,14.0,rsrp_n0c1_b1800,scn_b2779850236462be
5,n0c1,n0,-380.34,-489.80,30.0,120.0,b700,7.000000e+08,100000000.0,4.0,0.0,12.0,rsrp_n0c1_b700,scn_b2779850236462be
6,n0c2,n0,-380.34,-489.80,30.0,240.0,b2600,2.600000e+09,100000000.0,8.0,0.0,16.0,rsrp_n0c2_b2600,scn_b2779850236462be
7,n0c2,n0,-380.34,-489.80,30.0,240.0,b1800,1.800000e+09,100000000.0,6.0,0.0,14.0,rsrp_n0c2_b1800,scn_b2779850236462be
8,n0c2,n0,-380.34,-489.80,30.0,240.0,b700,7.000000e+08,100000000.0,4.0,0.0,12.0,rsrp_n0c2_b700,scn_b2779850236462be
9,n1c0,n1,-465.72,352.86,30.0,0.0,b2600,2.600000e+09,100000000.0,8.0,0.0,16.0,rsrp_n1c0_b2600,scn_b2779850236462be


In [8]:
# Bounds are per band, and wider for the higher bands. An optimizer stays inside
# them throughout; a baseline outside its own bounds is rejected by Tilt itself.
cells.groupby("band", observed=True)[
    ["tilt_baseline_deg", "tilt_min_deg", "tilt_max_deg"]
].agg(["min", "max"])

tilt_baseline_deg      tilt_min_deg      tilt_max_deg      
                    min  max          min  max          min   max
band                                                             
b1800               6.0  6.0          0.0  0.0         14.0  14.0
b2600               8.0  8.0          0.0  0.0         16.0  16.0
b700                4.0  4.0          0.0  0.0         12.0  12.0

## 5. The verified MDT

Same rows and same measurements as the interim CSV — nothing is dropped — but typed,
given a stable row order, and carrying a `reported_*` flag per cell-band beside its
`rsrp_*` and `sinr_*`.

The flag is the point. An empty measurement means *no path* **or** *censored by the
reporting model*, and the CSV cannot distinguish them. Carrying the indicator
explicitly stops a later reader treating the gap as a zero, and lets `02x` model the
censoring rather than impute over it.

In [ ]:
from src.data.build import build_mdt

mdt = build_mdt(artifacts)
measurement = [column for column in mdt.columns if column.startswith("rsrp_")]
sinr = [column for column in mdt.columns if column.startswith("sinr_")]
flags = [column for column in mdt.columns if column.startswith("reported_")]

print(f"{len(mdt):,} rows x {mdt.shape[1]} columns")
print(f"  {mdt.shape[1] - len(measurement) - len(sinr) - len(flags)} position, "
      f"{len(measurement)} RSRP, {len(sinr)} SINR, {len(flags)} reported")
mdt.head()


In [ ]:
pd.DataFrame(
    {
        "reported": [
            mdt[[c for c in flags if c.endswith(f"_{band}")]].to_numpy().mean()
            for band in artifacts.band_labels
        ],
        "median_rsrp_dbm": [
            np.nanmedian(mdt[[c for c in measurement if c.endswith(f"_{band}")]].to_numpy())
            for band in artifacts.band_labels
        ],
        "median_sinr_db": [
            np.nanmedian(mdt[[c for c in sinr if c.endswith(f"_{band}")]].to_numpy())
            for band in artifacts.band_labels
        ],
    },
    index=artifacts.band_labels,
)

## 6. Audit

What changed between the interim CSV and the processed table. Nothing should have
moved except dtypes and the added columns — this table is the evidence of that.

In [ ]:
raw = artifacts.mdt

pd.DataFrame(
    [
        ("rows", f"{len(raw):,}", f"{len(mdt):,}"),
        ("columns", raw.shape[1], mdt.shape[1]),
        ("float64 columns", int((raw.dtypes == "float64").sum()),
         int((mdt.dtypes == "float64").sum())),
        ("memory (MB)", round(raw.memory_usage(deep=True).sum() / 1e6, 2),
         round(mdt.memory_usage(deep=True).sum() / 1e6, 2)),
        ("measurements present", f"{raw[measurement].notna().to_numpy().mean():.1%}",
         f"{mdt[flags].to_numpy().mean():.1%}"),
        ("SINR present", f"{raw[sinr].notna().to_numpy().mean():.1%}",
         f"{mdt[sinr].notna().to_numpy().mean():.1%}"),
    ],
    columns=["property", "interim csv", "processed parquet"],
)

In [ ]:
# The invariants worth asserting rather than eyeballing: no row was lost, every
# flag really does mirror its measurement, and SINR is present exactly where RSRP
# is, so one flag covers both.
assert len(mdt) == len(raw), "preprocessing must not drop rows"
assert (mdt[flags].to_numpy() == mdt[measurement].notna().to_numpy()).all()
assert (mdt[sinr].notna().to_numpy() == mdt[measurement].notna().to_numpy()).all()
print("row count preserved; every reported_* mirrors its rsrp_* and its sinr_*")


## 7. Persist

Parquet rather than CSV: it round-trips dtypes, so the boolean flags stay boolean and
a missing measurement stays a real NaN instead of an empty string a reader has to
guess at.

Then track them with DVC so the exact contents are pinned to this Git commit:

```bash
dvc add data/processed/mdt.parquet data/processed/cell.parquet
git add data/processed/*.dvc configs/data.yaml
git commit -m "Build the processed MDT and cell tables"
```

In [13]:
from src.data.load import save

mdt_path = save(mdt, cfg.data.output.mdt_file)
cell_path = save(cells, cfg.data.output.cell_file)

pd.DataFrame(
    [
        {"file": str(path), "size_mb": round(Path(path).stat().st_size / 1e6, 3)}
        for path in (mdt_path, cell_path)
    ]
)

,file,size_mb
0,data\processed\mdt.parquet,0.818
1,data\processed\cell.parquet,0.010


In [ ]:
# Read both back and confirm the files hold exactly what was built. A dtype that
# does not survive the round trip is a bug that would otherwise surface in 03.
pd.testing.assert_frame_equal(pd.read_parquet(mdt_path), mdt)
pd.testing.assert_frame_equal(pd.read_parquet(cell_path), cells)
assert sorted(cells["rsrp_column"].astype(str)) == sorted(measurement)
assert sorted(cells["sinr_column"].astype(str)) == sorted(sinr)
print("both files round-trip identically; cell.parquet and mdt.parquet agree on columns")

## 8. Handoff checklist

- [ ] Every check in section 3 passed, and each names its source
- [ ] No row was dropped, and no measurement was imputed, scaled or encoded
- [ ] The −120 / −90 dBm thresholds were used to classify nothing and to filter nothing
- [ ] Every measurement carries an explicit `reported_*` flag
- [ ] `cell.parquet` holds one row per cell-band pair, at the baseline the map was solved at
- [ ] Both files round-trip and are `dvc add`-ed

**Still open — the split.** `01_eda.ipynb` section 11 calls this a blocker, and it
still is: one scenario is on disk, so the intended between-scenario split cannot be
made. `scenario_id` travels in both files so that the split can be made on it as soon
as `task simulation` has run for several seeds. A random row split is never the
fallback — UEs repeat tiles, and every UE in a tile reads the same radio-map values.